In [3]:
import os
from gensim.models import Word2Vec
import re
from collections import Counter

# Նախ ստուգենք ընթացիկ պանակը
print("Ընթացիկ պանակ:", os.getcwd())
print("\nՖայլեր և թղթապանակներ:")
print(os.listdir('.'))

# 1. Տվյալների բեռնում train և test պանակներից
def load_data(base_path, use_test=False):
    """
    base_path: 'ilur-news-corpus'
    use_test: True - օգտագործել և՛ train և՛ test
              False - միայն train
    """
    texts = []
    categories = ['accidents', 'culture', 'economy', 'politics', 'society', 'sport', 'weather']

    # Ստուգենք արդյոք գոյություն ունի base_path-ը
    if not os.path.exists(base_path):
        raise FileNotFoundError(f"Թղթապանակը չի գտնվել: {base_path}")

    print(f"\nՓանակի բովանդակությունը:")
    print(os.listdir(base_path))

    # Որոշենք որ պանակներն օգտագործել
    datasets = ['train']
    if use_test:
        datasets.append('test')

    total_files = 0

    for dataset in datasets:
        dataset_path = os.path.join(base_path, dataset)

        if not os.path.exists(dataset_path):
            print(f"⚠️ {dataset} պանակը չի գտնվել")
            continue

        print(f"\n{'='*50}")
        print(f"📂 {dataset.upper()} պանակ")
        print(f"{'='*50}")

        for category in categories:
            category_path = os.path.join(dataset_path, category)

            # Ստուգենք կատեգորիայի գոյությունը
            if not os.path.exists(category_path):
                print(f"⚠️ {category} կատեգորիան չի գտնվել")
                continue

            file_count = 0

            for filename in os.listdir(category_path):
                if filename.endswith('.txt'):
                    file_path = os.path.join(category_path, filename)
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            content = f.read()
                            if content.strip():  # Ստուգել դատարկ չլինելը
                                texts.append(content)
                                file_count += 1
                    except Exception as e:
                        print(f"\n⚠️ Սխալ {filename}-ը կարդալիս: {e}")

            total_files += file_count
            print(f"  ✅ {category:15s}: {file_count:4d} ֆայլ")

    print(f"\n{'='*50}")
    print(f"📊 ԸՆԴԱՄԵՆԸ: {total_files} ֆայլ, {len(texts)} տեքստ")
    print(f"{'='*50}")

    return texts

# 2. Տեքստի մաքրում
def preprocess_text(text):
    # Հեռացնել թվերը
    text = re.sub(r'\d+', '', text)

    # Հեռացնել կետադրական նշանները, պահել միայն հայերեն տառերը և բացատները
    text = re.sub(r'[^\u0530-\u058F\s]', ' ', text)

    # Հեռացնել ավելորդ բացատները
    text = re.sub(r'\s+', ' ', text)

    # Փոքրատառ դարձնել և բաժանել բառերի
    words = text.lower().split()

    # Հեռացնել շատ կարճ բառերը (1 տառ)
    words = [w for w in words if len(w) > 1]

    return words

# 3. Նախադասությունների պատրաստում
def prepare_sentences(texts):
    sentences = []
    for text in texts:
        # Բաժանել նախադասությունների (հայերեն և անգլերեն վերջակետեր)
        text_sentences = re.split(r'[.!?։]+', text)
        for sentence in text_sentences:
            words = preprocess_text(sentence)
            if len(words) > 2:  # Պահել միայն 2+ բառով նախադասությունները
                sentences.append(words)
    return sentences

# ======================================================================
# ՀԻՄՆԱԿԱՆ ԿՈԴ
# ======================================================================

# 4. Տվյալների բեռնում
# Փորձեք տարբեր տարբերակներ
possible_paths = [
    'ilur-news-corpus',
    './ilur-news-corpus',
    '../ilur-news-corpus',
]

texts = None
dataset_path = None

for path in possible_paths:
    if os.path.exists(path):
        print(f"\n✅ Գտնվեց: {path}")
        dataset_path = path
        # Օգտագործել ՄԻԱՅՆ train պանակը ուսուցման համար
        texts = load_data(path, use_test=False)
        break
else:
    print("\n❌ Dataset-ը չի գտնվել ավտոմատ։")
    manual_path = input("\nՄուտքագրեք dataset-ի ճանապարհը: ")
    if os.path.exists(manual_path):
        dataset_path = manual_path
        texts = load_data(manual_path, use_test=False)

# Եթե գտնվել է, շարունակենք
if texts and len(texts) > 0:
    print(f"\n📊 Ընդամենը {len(texts)} տեքստ բեռնվեց")

    # Ցույց տանք առաջին տեքստի օրինակը
    print(f"\n📄 Առաջին տեքստի սկիզբը:")
    print(texts[0][:200] + "...")

    # 5. Նախադասությունների պատրաստում
    print("\n🔄 Նախադասությունների պատրաստում...")
    sentences = prepare_sentences(texts)
    print(f"✅ {len(sentences)} նախադասություն պատրաստ է")

    # Օրինակ նախադասություններ
    print("\n📝 Օրինակ նախադասություններ (առաջին 5):")
    for i, sent in enumerate(sentences[:5]):
        print(f"{i+1}. {' '.join(sent[:15])}{'...' if len(sent) > 15 else ''}")

    # 6. Բառերի հաճախականության հաշվարկ
    print("\n🔄 Բառերի վերլուծություն...")
    word_freq = Counter([word for sentence in sentences for word in sentence])
    print(f"📊 Ընդամենը {len(word_freq)} յուրահատուկ բառ")
    print(f"📊 Ամենաշատ հանդիպող 20 բառերը:")
    for word, count in word_freq.most_common(20):
        print(f"  {word:20s}: {count:5d}")

    # 7. Հազվադեպ բառերի հեռացում (min_count=5)
    print("\n🔄 Հազվադեպ բառերի հեռացում (< 5 անգամ)...")
    rare_words = sum(1 for word, count in word_freq.items() if count < 5)
    print(f"⚠️ Հեռացվում է {rare_words} հազվադեպ բառ")

    sentences_filtered = [[word for word in sentence if word_freq[word] >= 5]
                          for sentence in sentences]
    sentences_filtered = [s for s in sentences_filtered if len(s) > 2]

    print(f"✅ {len(sentences_filtered)} նախադասություն ֆիլտրվեց")

    # 8. Word2Vec մոդելի ստեղծում և ուսուցում
    print("\n" + "="*70)
    print("🚀 WORD2VEC ՄՈԴԵԼԻ ՈՒՍՈՒՑՈՒՄ")
    print("="*70)
    print("\n📋 Պարամետրեր:")
    print(f"  • vector_size:     300")
    print(f"  • window:          5")
    print(f"  • min_count:       5")
    print(f"  • sg (skip-gram):  1")
    print(f"  • negative:        15")
    print(f"  • epochs:          20")
    print(f"  • workers:         4")
    print("\n⏳ Սպասեք, ուսուցումը կարող է տևել մի քանի րոպե...\n")

    model = Word2Vec(
        sentences=sentences_filtered,
        vector_size=300,
        window=5,
        min_count=5,
        sg=1,              # Skip-gram
        negative=15,       # Negative sampling
        epochs=20,
        workers=4
    )

    print("\n✅ Մոդելը ուսուցվեց հաջողությամբ!")

    # 9. Մոդելի պահպանում
    model_filename = "armenian_word2vec.model"
    model.save(model_filename)
    print(f"💾 Մոդելը պահպանվեց: {model_filename}")

    # Պահպանել նաև միայն word vectors-ները (ավելի փոքր ֆայլ)
    model.wv.save("armenian_word2vec.wordvectors")
    print(f"💾 Word vectors-ները պահպանվեցին: armenian_word2vec.wordvectors")

    # 10. Մոդելի վիճակագրություն
    print("\n" + "="*70)
    print("📊 ՄՈԴԵԼԻ ՎԻՃԱԿԱԳՐՈՒԹՅՈՒՆ")
    print("="*70)
    print(f"  • Բառարանի չափը:        {len(model.wv):,}")
    print(f"  • Vector չափը:           {model.wv.vector_size}")
    print(f"  • Ուսուցման նախադասություններ: {len(sentences_filtered):,}")

    # 11. Մոդելի թեստ
    print("\n" + "="*70)
    print("🧪 ՄՈԴԵԼԻ ԹԵՍՏ")
    print("="*70)

    # Ցույց տանք բառարանի մի քանի բառեր
    print(f"\n📚 Բառարանի առաջին 30 բառերը:")
    sample_words = list(model.wv.index_to_key[:30])
    for i, word in enumerate(sample_words, 1):
        print(f"{i:2d}. {word}", end="   ")
        if i % 5 == 0:
            print()
    print("\n")

    # Փորձենք գտնել նմանատիպ բառեր
    test_words = ['հայաստան', 'երևան', 'քաղաք', 'ժողովուրդ', 'երկիր',
                  'մարդիկ', 'աշխատանք', 'կյանք', 'պետք']

    print("\n🔍 Նմանատիպ բառերի որոնում:\n")

    found_words = []
    for word in test_words:
        if word in model.wv:
            found_words.append(word)
            print(f"'{word}'-ին նման բառեր:")
            try:
                similar = model.wv.most_similar(word, topn=5)
                for sim_word, score in similar:
                    print(f"  • {sim_word:20s} (նմանություն: {score:.3f})")
            except:
                print(f"  ⚠️ Չհաջողվեց գտնել նմանատիպ բառեր")
            print()

    if not found_words:
        print("⚠️ Թեստային բառերը չգտնվեցին բառարանում։")
        print("Փորձեք բառարանի առկա բառերից:")
        available = list(model.wv.index_to_key[100:110])
        print(available)

    print("\n" + "="*70)
    print("✅ ԱՄԲՈՂՋՈՒԹՅԱՄԲ ԱՎԱՐՏՎԱԾ!")
    print("="*70)

else:
    print("\n❌ Տեքստեր չբեռնվեցին։ Ստուգեք dataset-ի ճանապարհը։")

Ընթացիկ պանակ: C:\Users\narek\PycharmProjects\nlp-course-2025.1\Նարեկ Ստեփանյան\Լաբ10

Ֆայլեր և թղթապանակներ:
['ilur-news-corpus', 'w2v example with brown corpus.ipynb', 'w2v.ipynb', 'word_analogy_task_hy.txt']

✅ Գտնվեց: ilur-news-corpus

Փանակի բովանդակությունը:
['test', 'train']

📂 TRAIN պանակ
  ✅ accidents      : 1163 ֆայլ
  ✅ culture        :  796 ֆայլ
  ✅ economy        : 1653 ֆայլ
  ✅ politics       : 1303 ֆայլ
  ✅ society        : 1304 ֆայլ
  ✅ sport          : 2238 ֆայլ
  ✅ weather        : 1484 ֆայլ

📊 ԸՆԴԱՄԵՆԸ: 9941 ֆայլ, 9941 տեքստ

📊 Ընդամենը 9941 տեքստ բեռնվեց

📄 Առաջին տեքստի սկիզբը:

Փետրվարին Վայոց Ձորի Ջերմուկ քաղաքում արձանագրված սննդային թունավորման՝ բոտուլիզմի դեպքը մարտի 4-ին ավարտվել է մահվան ելքով, հայտնում է ՀՀ Առողջապահության նախարարությունը:


Սննդային թունավորման պատճ...

🔄 Նախադասությունների պատրաստում...
✅ 42486 նախադասություն պատրաստ է

📝 Օրինակ նախադասություններ (առաջին 5):
1. փետրվարին վայոց ձորի ջերմուկ քաղաքում արձանագրված սննդային թունավորման՝ բոտուլ